In [4]:
#r "C:\Users\user\Desktop\Remish\practice2026\task17\bin\Debug\net10.0\task17.dll"
#r "nuget: ScottPlot, 5.0.21"

using System;
using System.Diagnostics;
using System.Linq;
using System.Threading;
using System.Collections.Generic;
using System.Collections.Concurrent;
using System.IO;
using ScottPlot;
using task17;
using Microsoft.AspNetCore.Html;

public class SimpleCommand : ICommand
{
    private readonly Action _action;
    public SimpleCommand(Action action = null) => _action = action;
    public void Execute() => _action?.Invoke();
}

public class LongCommand : ILongCommand
{
    private int _remaining;
    private readonly Action _onExecute;
    public bool IsCompleted => _remaining <= 0;
    public string Name { get; }

    public LongCommand(int required, string name = "", Action onExecute = null)
    {
        _remaining = required;
        Name = name;
        _onExecute = onExecute;
    }

    public void Execute()
    {
        if (!IsCompleted)
        {
            _remaining--;
            _onExecute?.Invoke();
        }
    }
}


Console.WriteLine("ТЕСТ 1: Выполнение без планировщика (последовательно)");
var log1 = new List<string>();
var s1 = new RoundRobinScheduler();
var server1 = new ServerThread(s1);

server1.Start();
server1.Add(new SimpleCommand(() => { log1.Add("Команда 1"); Thread.Sleep(20); }));
server1.Add(new SimpleCommand(() => { log1.Add("Команда 2"); Thread.Sleep(20); }));
server1.Add(new SimpleCommand(() => { log1.Add("Команда 3"); Thread.Sleep(20); }));
Thread.Sleep(200);
server1.HardStop();
server1.Join();

Console.WriteLine("Порядок выполнения: " + string.Join(" -> ", log1));
Console.WriteLine();

Console.WriteLine("ТЕСТ 2: Выполнение с планировщиком (Round Robin)");
var log2 = new List<string>();
var s2 = new RoundRobinScheduler();
var server2 = new ServerThread(s2);

server2.Start();
server2.Add(new LongCommand(3, "A", () => log2.Add("A")));
server2.Add(new LongCommand(3, "B", () => log2.Add("B")));
server2.Add(new LongCommand(3, "C", () => log2.Add("C")));
Thread.Sleep(200);
server2.HardStop();
server2.Join();

Console.WriteLine("Порядок выполнения: " + string.Join(" -> ", log2));
Console.WriteLine($"A: {log2.Count(x => x == "A")}, B: {log2.Count(x => x == "B")}, C: {log2.Count(x => x == "C")}");
Console.WriteLine();

Console.WriteLine("ТЕСТ 3: Сравнение времени выполнения");

var projectPath = Directory.GetCurrentDirectory();
var logPath = Path.Combine(projectPath, "benchmark_results.txt");
var logLines = new List<string>();
logLines.Add("СРАВНЕНИЕ: без планировщика vs с планировщиком");
logLines.Add("");

var counts = new List<int>();
var withoutPlanner = new List<double>();
var withPlanner = new List<double>();

for (int n = 1; n <= 10; n++)
{
    int stepsPerTask = 10;
    
    var sw1 = Stopwatch.StartNew();
    var bs1 = new RoundRobinScheduler();
    var bserver1 = new ServerThread(bs1);
    int done1 = 0;
    bserver1.Start();
    
    for (int i = 0; i < n; i++)
    {
        bserver1.Add(new SimpleCommand(() =>
        {
            for (int s = 0; s < stepsPerTask; s++)
            {
                Interlocked.Increment(ref done1);
                Thread.Sleep(5);
            }
        }));
    }
    
    while (done1 < n * stepsPerTask) Thread.Sleep(1);
    bserver1.HardStop();
    bserver1.Join();
    sw1.Stop();

    var sw2 = Stopwatch.StartNew();
    var bs2 = new RoundRobinScheduler();
    var bserver2 = new ServerThread(bs2);
    int done2 = 0;
    bserver2.Start();
    
    for (int i = 0; i < n; i++)
    {
        bserver2.Add(new LongCommand(stepsPerTask, onExecute: () =>
        {
            Interlocked.Increment(ref done2);
            Thread.Sleep(5);
        }));
    }
    
    while (done2 < n * stepsPerTask) Thread.Sleep(1);
    bserver2.HardStop();
    bserver2.Join();
    sw2.Stop();

    counts.Add(n);
    withoutPlanner.Add(sw1.Elapsed.TotalMilliseconds);
    withPlanner.Add(sw2.Elapsed.TotalMilliseconds);
    
    var line = $"Задач: {n,2} | Без планировщика: {sw1.Elapsed.TotalMilliseconds,6:F0} мс | С планировщиком: {sw2.Elapsed.TotalMilliseconds,6:F0} мс";
    logLines.Add(line);
    Console.WriteLine(line);
}

logLines.Add("");
logLines.Add($"Тест завершён: {DateTime.Now}");
File.WriteAllLines(logPath, logLines);
Console.WriteLine($"\nРезультаты записаны в: {logPath}");

var plot = new Plot();
plot.Title("Время выполнения");
plot.XLabel("Кол-во задач");
plot.YLabel("Время (мс)");

plot.Add.Scatter(counts.Select(x => (double)x).ToArray(), withoutPlanner.ToArray())
    .Label = "Без планировщика ";
plot.Add.Scatter(counts.Select(x => (double)x).ToArray(), withPlanner.ToArray())
    .Label = "Round Robin";

plot.ShowLegend();

var fn = Path.Combine(projectPath, "graph.png");
plot.SavePng(fn, 800, 600);
display(HTML($"<img src='{fn}?t={DateTime.Now.Ticks}' width='700'/>"));

Installed Packages ScottPlot, 5.0.21

ТЕСТ 1: Выполнение без планировщика (последовательно)
Порядок выполнения: Команда 1 -> Команда 2 -> Команда 3

ТЕСТ 2: Выполнение с планировщиком (Round Robin)
Порядок выполнения: A -> B -> A -> C -> B -> A -> C -> B -> C
A: 3, B: 3, C: 3

ТЕСТ 3: Сравнение времени выполнения
Задач:  1 | Без планировщика:    151 мс | С планировщиком:    164 мс
Задач:  2 | Без планировщика:    316 мс | С планировщиком:    312 мс
Задач:  3 | Без планировщика:    456 мс | С планировщиком:    436 мс
Задач:  4 | Без планировщика:    410 мс | С планировщиком:    604 мс
Задач:  5 | Без планировщика:    750 мс | С планировщиком:    782 мс
Задач:  6 | Без планировщика:    890 мс | С планировщиком:    778 мс
Задач:  7 | Без планировщика:   1150 мс | С планировщиком:   1132 мс
Задач:  8 | Без планировщика:   1318 мс | С планировщиком:   1316 мс
Задач:  9 | Без планировщика:   1233 мс | С планировщиком:   1201 мс
Задач: 10 | Без планировщика:   1465 мс | С планировщиком:   1665 мс

Результаты записаны в: c:\Users\